In [ ]:
import pandas as pd 
from great_tables import GT, style, loc

In [ ]:
df = pd.read_csv ("/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/Clean Tables(Table2 (simplified poster)).csv", keep_default_na=False)
df.head()

In [ ]:
df2 = pd.read_csv("/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/Clean Tables(Table3 (simplified poster)) (2).csv", keep_default_na = False)
df2.head()

In [ ]:
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This table is for non-lag data (original results where we were just assessing whether there were any associations
# between bacterial infections and NDDs, independent of time).


# ============================================================
# EDIT THIS SECTION
# ============================================================

input_df = df
output_file = "clean_three_cohort_table.png"
preview_only = False   # True = show on screen, False = save to file

# Your actual dataframe column names
NDD_header = "NDD"
condition_header = "Description"
cohort_header = "Cohort"
HR_header = "OR/HR"
CI_min_header = "CI Min"
CI_max_header = "CI Max"
N_pairs_header = "N Pairs"
p_value_header = "P Value, FDR"

# Three cohorts to show
cohorts_to_show = ["FinnGen", "AoU", "UKB"]

# Optional cleanup for cohort names
cohort_value_map = {
    "FinnGen": "FinnGen",
    "Finngen": "FinnGen",
    "finnGen": "FinnGen",
    "AoU": "AoU",
    "AOU": "AoU",
    "aou": "AoU",
    "All of Us": "AoU",
    "UKB": "UKB",
    "ukb": "UKB",
    "UK Biobank": "UKB",
}

# NDD display cleanup
ndd_name_map = {
    "VAS": "VAS",
    "PD": "PD",
    "AD": "AD",
    "Dementia": "DEM",
    "Vascular Dementia": "VAS",
    "Parkinson's Disease": "PD",
    "Alzheimer's Disease": "AD",
    "Vascular dementia": "VAS",
    "Vascular Dementia ": "VAS",
}

# Formatting
decimal_places = 2
p_value_decimals = 2
p_value_label = "FDR"
wrap_condition_at = 45

# Image settings
figure_width = 30
row_height = 0.72
font_size = 17
dpi = 600


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_blank_cells(series):
    return series.replace(r"^\s*$", np.nan, regex=True)


def format_number(x, digits=2):
    if pd.isna(x) or str(x).strip() == "":
        return ""
    try:
        return f"{float(x):.{digits}f}"
    except Exception:
        return str(x)


def format_integer(x):
    if pd.isna(x) or str(x).strip() == "":
        return ""
    try:
        return f"{int(float(x)):,}"
    except Exception:
        return str(x)


def format_pvalue(x, digits=2):
    if pd.isna(x) or str(x).strip() == "":
        return ""

    try:
        val = float(str(x).strip())
    except Exception:
        return str(x)

    if val == 0:
        return "0"

    formatted = f"{val:.{digits}e}"
    mantissa, exponent = formatted.split("e")
    mantissa = mantissa.rstrip("0").rstrip(".")
    exponent = str(int(exponent))
    return f"{mantissa}e{exponent}"


def make_metric_texts(row):
    hr = format_number(row["_HR"], decimal_places)
    ci_min = format_number(row["_CI_min"], decimal_places)
    ci_max = format_number(row["_CI_max"], decimal_places)
    n_pairs = format_integer(row["_N_pairs"])
    p_value = format_pvalue(row["_P_value"], p_value_decimals)

    if hr == "":
        return "", "", ""

    return f"{hr} ({ci_min}–{ci_max})", n_pairs, p_value


def prepare_clean_table(df):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()

    column_map = {
        NDD_header: "_NDD",
        condition_header: "_Condition",
        cohort_header: "_Cohort",
        HR_header: "_HR",
        CI_min_header: "_CI_min",
        CI_max_header: "_CI_max",
        N_pairs_header: "_N_pairs",
        p_value_header: "_P_value",
    }

    missing_cols = [col for col in column_map if col not in df.columns]
    if missing_cols:
        print("Missing columns:")
        for col in missing_cols:
            print(f"  - {col}")

        print("\nAvailable columns in your dataframe:")
        for col in df.columns:
            print(f"  - {col}")

        raise ValueError("Update the *_header variables at the top of the script.")

    df = df.rename(columns=column_map)

    # Fill down repeated labels
    df["_NDD"] = clean_blank_cells(df["_NDD"]).ffill()
    df["_Condition"] = clean_blank_cells(df["_Condition"]).ffill()

    # Optional NDD cleanup / relabeling
    df["_NDD"] = df["_NDD"].astype(str).str.strip()
    df["_NDD"] = df["_NDD"].replace(ndd_name_map)

    # Clean cohort names
    df["_Cohort"] = df["_Cohort"].astype(str).str.strip()
    df["_Cohort"] = df["_Cohort"].replace(cohort_value_map)

    # Keep only requested cohorts
    df = df[df["_Cohort"].isin(cohorts_to_show)].copy()

    index_cols = ["_NDD", "_Condition"]
    wide = df[index_cols].drop_duplicates().copy()

    for cohort in cohorts_to_show:
        cohort_df = (
            df[df["_Cohort"] == cohort]
            .drop_duplicates(subset=index_cols, keep="first")
            [index_cols + ["_HR", "_CI_min", "_CI_max", "_N_pairs", "_P_value"]]
            .copy()
        )

        cohort_df[[f"{cohort} HR_CI", f"{cohort} Pairs", f"{cohort} FDR"]] = cohort_df.apply(
            lambda row: pd.Series(make_metric_texts(row)),
            axis=1
        )

        cohort_df = cohort_df[
            index_cols + [f"{cohort} HR_CI", f"{cohort} Pairs", f"{cohort} FDR"]
        ]

        wide = wide.merge(cohort_df, on=index_cols, how="left")

    metric_cols = [c for c in wide.columns if c not in index_cols]
    for col in metric_cols:
        wide[col] = wide[col].fillna("")

    return wide


def build_display_rows(wide):
    display_rows = []
    group_start_rows = []

    previous_ndd = None
    previous_condition = None

    metric_cols = []
    for cohort in cohorts_to_show:
        metric_cols.extend([
            f"{cohort} HR_CI",
            f"{cohort} Pairs",
            f"{cohort} FDR",
        ])

    for _, row in wide.iterrows():
        ndd = str(row["_NDD"])
        condition = str(row["_Condition"])

        show_ndd = ndd != previous_ndd
        show_condition = condition != previous_condition or show_ndd

        # +2 because table has 2 header rows
        if show_ndd:
            group_start_rows.append(len(display_rows) + 2)

        row_out = [
            ndd if show_ndd else "",
            textwrap.fill(condition, width=wrap_condition_at) if show_condition else "",
        ]

        for col in metric_cols:
            row_out.append(row[col])

        display_rows.append(row_out)

        previous_ndd = ndd
        previous_condition = condition

    return display_rows, group_start_rows


def save_table_image(display_rows, group_start_rows):
    # Two-level header
    header_top = ["", ""]
    header_bottom = ["NDD", "Associated condition"]

    for cohort in cohorts_to_show:
        header_top.extend(["", "", ""])
        header_bottom.extend(["HR (95% CI)", "Pairs", f"{p_value_label} p"])

    table_rows = [header_top, header_bottom] + display_rows

    n_rows = len(table_rows)
    n_cols = len(table_rows[0])

    fig_height = max(2.6, row_height * (len(display_rows) + 2.25))
    fig, ax = plt.subplots(figsize=(figure_width, fig_height))
    ax.axis("off")

    table = ax.table(
        cellText=table_rows,
        cellLoc="left",
        colLoc="left",
        loc="upper left",
        bbox=[0.01, 0.02, 0.98, 0.96],
    )

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.scale(1.1, 1.35)

    # Wider columns
    col_widths = [
        0.12,  # NDD
        0.29,  # Associated condition
        0.16,  # Cohort 1 HR (95% CI)
        0.05,  # Cohort 1 Pairs
        0.07,  # Cohort 1 FDR p
        0.16,  # Cohort 2 HR (95% CI)
        0.05,  # Cohort 2 Pairs
        0.07,  # Cohort 2 FDR p
        0.16,  # Cohort 3 HR (95% CI)
        0.05,  # Cohort 3 Pairs
        0.07,  # Cohort 3 FDR p
    ]

    for col_idx, width in enumerate(col_widths):
        for row_idx in range(n_rows):
            table[row_idx, col_idx].set_width(width)

    # Make cells open, not boxed
    for (row_idx, col_idx), cell in table.get_celld().items():
        cell.PAD = 0.02
        cell.set_edgecolor("white")
        cell.set_linewidth(0.0)

        txt = cell.get_text()
        txt.set_color("#1F2937")
        txt.set_va("center")

        if row_idx == 0:
            cell.set_facecolor("white")
            txt.set_weight("bold")
            txt.set_ha("center" if col_idx >= 2 else "left")

        elif row_idx == 1:
            cell.set_facecolor("white")
            txt.set_weight("bold")
            txt.set_ha("center" if col_idx >= 2 else "left")

        else:
            cell.set_facecolor("white")
            txt.set_ha("left" if col_idx in [0, 1] else "center")

            data_row_idx = row_idx - 2
            if col_idx == 0 and display_rows[data_row_idx][0] != "":
                txt.set_weight("bold")

    # Draw clean horizontal rules only
    fig.canvas.draw()

    left_x = table[0, 0].get_x()
    right_x = table[0, n_cols - 1].get_x() + table[0, n_cols - 1].get_width()

    top_y = table[0, 0].get_y() + table[0, 0].get_height()
    header_bottom_y = table[1, 0].get_y()
    bottom_y = table[n_rows - 1, 0].get_y()

    ax.hlines(top_y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.1)
    ax.hlines(bottom_y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.1)
    ax.hlines(header_bottom_y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.0)

    # Group separators
    for row_idx in group_start_rows:
        if row_idx < 2:
            continue
        y = table[row_idx, 0].get_y() + table[row_idx, 0].get_height()
        ax.hlines(y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.0)

    # Vertical separators:
    # between condition and cohort area, and between cohort groups
    # cohort 1 starts at col 2, cohort 2 at col 5, cohort 3 at col 8
    sep1_x = table[0, 2].get_x()
    sep2_x = table[0, 5].get_x()
    sep3_x = table[0, 8].get_x()

    ax.vlines(sep1_x, bottom_y, top_y, transform=ax.transAxes, color="#98A2B3", linewidth=0.9)
    ax.vlines(sep2_x, bottom_y, top_y, transform=ax.transAxes, color="#98A2B3", linewidth=0.9)
    ax.vlines(sep3_x, bottom_y, top_y, transform=ax.transAxes, color="#98A2B3", linewidth=0.9)

    # Cohort labels centered over their 3-column spans
    start_col = 2
    for cohort in cohorts_to_show:
        end_col = start_col + 2
        x0 = table[0, start_col].get_x()
        x1 = table[0, end_col].get_x() + table[0, end_col].get_width()
        y0 = table[0, start_col].get_y()
        h = table[0, start_col].get_height()

        ax.text(
            (x0 + x1) / 2,
            y0 + h / 2,
            cohort,
            ha="center",
            va="center",
            fontsize=font_size,
            fontweight="bold",
            transform=ax.transAxes,
            color="#1F2937",
        )

        start_col += 3

    if preview_only:
        plt.show()
    else:
        output_file = '/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/Tables/table_no_lag.png'
        plt.savefig(output_file, dpi=dpi, bbox_inches="tight", pad_inches=0.05)
        print(f"Saved: {output_file}")

    plt.close()


# ============================================================
# RUN
# ============================================================

wide = prepare_clean_table(input_df)
display_rows, group_start_rows = build_display_rows(wide)
save_table_image(display_rows, group_start_rows)

In [ ]:
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



#This table is for  lags 

input_df = df2
output_file = "clean_cohort_table.png"
preview_only = True   # True = show on screen, False = save to file

# Your actual dataframe column names
NDD_header = "NDD"
condition_header = "Description"
cohort_header = "Cohort"
lag_header = "Lag"
HR_header = "HR"
CI_min_header = "CI Min"
CI_max_header = "CI Max"
N_pairs_header = "N Pairs"
p_value_header = "P Value, FDR"

# Cohorts to show
cohorts_to_show = ["FinnGen", "AoU"]

# Optional cleanup for cohort names
cohort_value_map = {
    "FinnGen": "FinnGen",
    "Finngen": "FinnGen",
    "finnGen": "FinnGen",
    "AoU": "AoU",
    "AOU": "AoU",
    "aou": "AoU",
    "All of Us": "AoU",
}

# Formatting
decimal_places = 2
p_value_decimals = 2
p_value_label = "FDR"
wrap_condition_at = 45

# Image settings
figure_width = 30
row_height = 0.78
font_size = 15
dpi = 300


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def clean_blank_cells(series):
    return series.replace(r"^\s*$", np.nan, regex=True)


def format_number(x, digits=2):
    if pd.isna(x) or str(x).strip() == "":
        return ""
    try:
        return f"{float(x):.{digits}f}"
    except Exception:
        return str(x)


def format_integer(x):
    if pd.isna(x) or str(x).strip() == "":
        return ""
    try:
        return f"{int(float(x)):,}"
    except Exception:
        return str(x)


def format_pvalue(x, digits=2):
    if pd.isna(x) or str(x).strip() == "":
        return ""

    try:
        val = float(str(x).strip())
    except Exception:
        return str(x)

    if val == 0:
        return "0"

    formatted = f"{val:.{digits}e}"
    mantissa, exponent = formatted.split("e")
    mantissa = mantissa.rstrip("0").rstrip(".")
    exponent = str(int(exponent))
    return f"{mantissa}e{exponent}"


def make_metric_texts(row):
    hr = format_number(row["_HR"], decimal_places)
    ci_min = format_number(row["_CI_min"], decimal_places)
    ci_max = format_number(row["_CI_max"], decimal_places)
    n_pairs = format_integer(row["_N_pairs"])
    p_value = format_pvalue(row["_P_value"], p_value_decimals)

    if hr == "":
        return "—", "—", "—"

    return f"{hr} ({ci_min}–{ci_max})", n_pairs, p_value


def prepare_clean_table(df):
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()

    column_map = {
        NDD_header: "_NDD",
        condition_header: "_Condition",
        cohort_header: "_Cohort",
        lag_header: "_Lag",
        HR_header: "_HR",
        CI_min_header: "_CI_min",
        CI_max_header: "_CI_max",
        N_pairs_header: "_N_pairs",
        p_value_header: "_P_value",
    }

    ndd_name_map = {
    "Vascular dementia": "Vascular Dementia",
    "PD": "Parkinson's Disease",
    "AD": "Alzheimer's Disease",
    "Dementia": "Dementia",
    }

    missing_cols = [col for col in column_map if col not in df.columns]
    if missing_cols:
        print("Missing columns:")
        for col in missing_cols:
            print(f"  - {col}")

        print("\nAvailable columns in your dataframe:")
        for col in df.columns:
            print(f"  - {col}")

        raise ValueError("Update the *_header variables at the top of the script.")


    df = df.rename(columns=column_map)

    # Fill down repeated labels
    df["_NDD"] = clean_blank_cells(df["_NDD"]).ffill()
    df["_NDD"] = df["_NDD"].replace(ndd_name_map)
    df["_Condition"] = clean_blank_cells(df["_Condition"]).ffill()

    # Clean cohort names
    df["_Cohort"] = df["_Cohort"].astype(str).str.strip()
    df["_Cohort"] = df["_Cohort"].replace(cohort_value_map)

    # Keep only requested cohorts
    df = df[df["_Cohort"].isin(cohorts_to_show)].copy()

    index_cols = ["_NDD", "_Condition", "_Lag"]
    wide = df[index_cols].drop_duplicates().copy()

    for cohort in cohorts_to_show:
        cohort_df = (
            df[df["_Cohort"] == cohort]
            .drop_duplicates(subset=index_cols, keep="first")
            [index_cols + ["_HR", "_CI_min", "_CI_max", "_N_pairs", "_P_value"]]
            .copy()
        )

        cohort_df[[f"{cohort} HR_CI", f"{cohort} Pairs", f"{cohort} FDR"]] = cohort_df.apply(
            lambda row: pd.Series(make_metric_texts(row)),
            axis=1
        )

        cohort_df = cohort_df[
            index_cols + [f"{cohort} HR_CI", f"{cohort} Pairs", f"{cohort} FDR"]
        ]

        wide = wide.merge(cohort_df, on=index_cols, how="left")

    metric_cols = [c for c in wide.columns if c not in index_cols]
    for col in metric_cols:
        if col not in index_cols:
            wide[col] = wide[col].fillna("—")

    return wide


def build_display_rows(wide):
    display_rows = []
    group_start_rows = []

    previous_ndd = None
    previous_condition = None

    metric_cols = []
    for cohort in cohorts_to_show:
        metric_cols.extend([
            f"{cohort} HR_CI",
            f"{cohort} Pairs",
            f"{cohort} FDR",
        ])

    for _, row in wide.iterrows():
        ndd = str(row["_NDD"])
        condition = str(row["_Condition"])
        lag = row["_Lag"]

        show_ndd = ndd != previous_ndd
        show_condition = condition != previous_condition or show_ndd

        if show_ndd:
            # +2 because table has 2 header rows
            group_start_rows.append(len(display_rows) + 2)

        try:
            lag_float = float(lag)
            lag_text = str(int(lag_float)) if lag_float.is_integer() else str(lag)
        except Exception:
            lag_text = str(lag)

        row_out = [
            ndd if show_ndd else "",
            textwrap.fill(condition, width=wrap_condition_at) if show_condition else "",
            lag_text,
        ]

        for col in metric_cols:
            row_out.append(row[col])

        display_rows.append(row_out)

        previous_ndd = ndd
        previous_condition = condition

    return display_rows, group_start_rows


def save_table_image(display_rows, group_start_rows):
    # Two-level header
    header_top = ["", "", ""]
    header_bottom = ["NDD", "Associated condition", "Lag"]

    for cohort in cohorts_to_show:
        header_top.extend(["", "", ""])
        header_bottom.extend(["HR (95% CI)", "Pairs", f"{p_value_label} p"])

    table_rows = [header_top, header_bottom] + display_rows

    n_rows = len(table_rows)
    n_cols = len(table_rows[0])

    fig_height = max(2.6, row_height * (len(display_rows) + 2.25))
    fig, ax = plt.subplots(figsize=(figure_width, fig_height))
    ax.axis("off")

    table = ax.table(
        cellText=table_rows,
        cellLoc="left",
        colLoc="left",
        loc="upper left",
        bbox=[0.01, 0.02, 0.98, 0.96],
    )

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.scale(1, 1.16)

    # Wider columns, especially HR/CI
    col_widths = [
        0.11,  # NDD
        0.24,  # Associated condition
        0.05,  # Lag
        0.18,  # FinnGen HR (95% CI)
        0.05,  # FinnGen Pairs
        0.07,  # FinnGen FDR p
        0.18,  # AoU HR (95% CI)
        0.05,  # AoU Pairs
        0.07,  # AoU FDR p
    ]

    for col_idx, width in enumerate(col_widths):
        for row_idx in range(n_rows):
            table[row_idx, col_idx].set_width(width)

    for (row_idx, col_idx), cell in table.get_celld().items():
        cell.PAD = 0.035
        cell.set_edgecolor("white")
        cell.set_linewidth(0.0)

        if row_idx == 0:
            cell.set_facecolor("white")
            cell.set_text_props(weight="bold", color="#1F2937")
        elif row_idx == 1:
            cell.set_facecolor("white")
            cell.set_text_props(weight="bold", color="#1F2937")
        else:
            cell.set_facecolor("white")
            cell.set_text_props(color="#1F2937")

            data_row_idx = row_idx - 2
            if col_idx == 0 and display_rows[data_row_idx][0] != "":
                cell.set_text_props(weight="bold", color="#1F2937")

        if col_idx >= 2:
            cell.set_text_props(ha="center")

    fig.canvas.draw()

    left_x = table[0, 0].get_x()
    right_x = table[0, n_cols - 1].get_x() + table[0, n_cols - 1].get_width()

    top_y = table[0, 0].get_y() + table[0, 0].get_height()
    header_bottom_y = table[1, 0].get_y()
    bottom_y = table[n_rows - 1, 0].get_y()

    # Outer border lines
    ax.hlines(top_y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.1)
    ax.hlines(bottom_y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.1)

    # Under header
    ax.hlines(header_bottom_y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.0)

    # Group separators
    for row_idx in group_start_rows:
        if row_idx < 2:
            continue
        y = table[row_idx, 0].get_y() + table[row_idx, 0].get_height()
        ax.hlines(y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.0)

    # FinnGen columns are 3,4,5. AoU starts at 6.
    sep_x = table[0, 6].get_x()
    ax.vlines(sep_x, bottom_y, top_y, transform=ax.transAxes, color="#98A2B3", linewidth=0.9)

    # Vertical line between Lag and FinnGen
    sep_x = table[0, 3].get_x()
    ax.vlines(sep_x,bottom_y,top_y,transform=ax.transAxes,color="#98A2B3", linewidth=0.9,)

    #cohort labels
    start_col = 3
    for cohort in cohorts_to_show:
        end_col = start_col + 2
        x0 = table[0, start_col].get_x()
        x1 = table[0, end_col].get_x() + table[0, end_col].get_width()
        y0 = table[0, start_col].get_y()
        h = table[0, start_col].get_height()

        ax.text(
            (x0 + x1) / 2,
            y0 + h / 2,
            cohort,
            ha="center",
            va="center",
            fontsize=font_size,
            fontweight="bold",
            transform=ax.transAxes,
            color="#1F2937",
        )

        start_col += 3

    if preview_only:
        plt.show()
    else:
        output_file = "/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/Tables/table_all_lags"
        plt.savefig(output_file, dpi=dpi, bbox_inches="tight", pad_inches=0.05)
        print(f"Saved: {output_file}")

    plt.close()


# ============================================================
# RUN
# ============================================================

wide = prepare_clean_table(input_df)
display_rows, group_start_rows = build_display_rows(wide)
save_table_image(display_rows, group_start_rows)

In [ ]:
icd10df = pd.read_csv('/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/icd10_simplified.csv', keep_default_na=False,encoding="cp1252")
icd10df.head()

In [ ]:
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


input_df = df2
output_file = "clean_table_ndd_icd10.png"
preview_only = False   # True = show on screen, False = save to file

# Your actual dataframe column names
NDD_header = "NDD"
icd10_header = "ICD10"
condition_header = "Description"
HR_header = "HR"
CI_min_header = "CI Min"
CI_max_header = "CI Max"
N_pairs_header = "N pairs"
p_value_header = "P, FDR"

# Formatting
decimal_places = 2
p_value_decimals = 2
p_value_label = "FDR"
wrap_condition_at = 45

# Figure settings
figure_width = 15.5
row_height = 0.72
font_size = 8.0
dpi = 300


# ============================================================
# HELPERS
# ============================================================

def clean_blank_cells(series):
    return series.replace(r"^\s*$", np.nan, regex=True)


def format_number(x, digits=2):
    if pd.isna(x) or str(x).strip() == "":
        return ""
    try:
        return f"{float(x):.{digits}f}"
    except Exception:
        return str(x)


def format_integer(x):
    if pd.isna(x) or str(x).strip() == "":
        return ""
    try:
        return f"{int(float(x)):,}"
    except Exception:
        return str(x)


def format_pvalue(x, digits=2):
    if pd.isna(x) or str(x).strip() == "":
        return ""

    try:
        val = float(str(x).strip())
    except Exception:
        return str(x)

    if val == 0:
        return "0"

    formatted = f"{val:.{digits}e}"
    mantissa, exponent = formatted.split("e")
    mantissa = mantissa.rstrip("0").rstrip(".")
    exponent = str(int(exponent))
    return f"{mantissa}e{exponent}"


def make_hr_ci_text(row):
    hr = format_number(row["_HR"], decimal_places)
    ci_min = format_number(row["_CI_min"], decimal_places)
    ci_max = format_number(row["_CI_max"], decimal_places)

    if hr == "":
        return ""

    return f"{hr} ({ci_min}–{ci_max})"


def prepare_table(df):
    df = icd10df.copy()
    df.columns = df.columns.astype(str).str.strip()

    column_map = {
        NDD_header: "_NDD",
        icd10_header: "_ICD10",
        condition_header: "_Condition",
        HR_header: "_HR",
        CI_min_header: "_CI_min",
        CI_max_header: "_CI_max",
        N_pairs_header: "_N_pairs",
        p_value_header: "_P_value",
    }

    missing_cols = [col for col in column_map if col not in df.columns]
    if missing_cols:
        print("Missing columns:")
        for col in missing_cols:
            print(f"  - {col}")

        print("\nAvailable columns in your dataframe:")
        for col in df.columns:
            print(f"  - {col}")

        raise ValueError("Update the header variables at the top of the script.")

    df = df.rename(columns=column_map)

    # Fill down repeated labels
    df["_NDD"] = clean_blank_cells(df["_NDD"]).ffill()
    df["_ICD10"] = clean_blank_cells(df["_ICD10"]).ffill()
    df["_Condition"] = clean_blank_cells(df["_Condition"]).ffill()

    # Build display columns
    df["HR (95% CI)"] = df.apply(make_hr_ci_text, axis=1)
    df["Pairs"] = df["_N_pairs"].apply(format_integer)
    df[f"{p_value_label} p"] = df["_P_value"].apply(lambda x: format_pvalue(x, p_value_decimals))

    display_df = df[[
        "_NDD",
        "_ICD10",
        "_Condition",
        "HR (95% CI)",
        "Pairs",
        f"{p_value_label} p",
    ]].copy()

    display_df = display_df.rename(columns={
        "_NDD": "NDD",
        "_ICD10": "ICD10 code",
        "_Condition": "Associated condition",
    })

    return display_df


def build_display_rows(display_df):
    display_rows = []
    group_start_rows = []

    previous_ndd = None
    previous_icd10 = None
    previous_condition = None

    for _, row in display_df.iterrows():
        ndd = str(row["NDD"])
        icd10 = str(row["ICD10 code"])
        condition = str(row["Associated condition"])

        show_ndd = ndd != previous_ndd
        show_icd10 = icd10 != previous_icd10 or show_ndd
        show_condition = condition != previous_condition or show_icd10

        if show_ndd:
            # +1 because there is one header row
            group_start_rows.append(len(display_rows) + 1)

        row_out = [
            ndd if show_ndd else "",
            icd10 if show_icd10 else "",
            textwrap.fill(condition, width=wrap_condition_at) if show_condition else "",
            row["HR (95% CI)"],
            row["Pairs"],
            row[f"{p_value_label} p"],
        ]

        display_rows.append(row_out)

        previous_ndd = ndd
        previous_icd10 = icd10
        previous_condition = condition

    return display_rows, group_start_rows


def save_table_image(display_rows, group_start_rows):
    headers = [
        "NDD",
        "ICD10 code",
        "Associated condition",
        "HR (95% CI)",
        "Pairs",
        f"{p_value_label} p",
    ]

    table_rows = [headers] + display_rows

    n_rows = len(table_rows)
    n_cols = len(headers)

    fig_height = max(2.6, row_height * (len(display_rows) + 1.8))
    fig, ax = plt.subplots(figsize=(figure_width, fig_height))
    ax.axis("off")

    table = ax.table(
        cellText=table_rows,
        cellLoc="left",
        colLoc="left",
        loc="upper left",
        bbox=[0.01, 0.02, 0.98, 0.96],
    )

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.scale(1, 1.16)

    # Column widths
    col_widths = [
        0.07,  # NDD
        0.07,  # ICD10 code
        0.12,  # Associated condition
        0.04,  # HR (95% CI)
        0.03,  # Pairs
        0.03,  # FDR p
    ]

    for col_idx, width in enumerate(col_widths):
        for row_idx in range(n_rows):
            table[row_idx, col_idx].set_width(width)

    # Style cells
    for (row_idx, col_idx), cell in table.get_celld().items():
        cell.PAD = 0.035
        cell.set_edgecolor("white")
        cell.set_linewidth(0.0)

        if row_idx == 0:
            cell.set_facecolor("white")
            cell.set_text_props(weight="bold", color="#1F2937")
        else:
            cell.set_facecolor("white")
            cell.set_text_props(color="#1F2937")

            data_row_idx = row_idx - 1
            if col_idx == 0 and display_rows[data_row_idx][0] != "":
                cell.set_text_props(weight="bold", color="#1F2937")

        if col_idx >= 3:
            cell.set_text_props(ha="center")

    # Horizontal rules only
    fig.canvas.draw()

    left_x = table[0, 0].get_x()
    right_x = table[0, n_cols - 1].get_x() + table[0, n_cols - 1].get_width()

    top_y = table[0, 0].get_y() + table[0, 0].get_height()
    header_bottom_y = table[0, 0].get_y()
    bottom_y = table[n_rows - 1, 0].get_y()

    ax.hlines(top_y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.1)
    ax.hlines(bottom_y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.1)
    ax.hlines(header_bottom_y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.0)

    for row_idx in group_start_rows:
        if row_idx <= 1:
            continue
        y = table[row_idx, 0].get_y() + table[row_idx, 0].get_height()
        ax.hlines(y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.0)

    if preview_only:
        plt.show()
    else:
        plt.savefig('/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/Tables/icd10table', dpi=dpi, bbox_inches="tight", pad_inches=0.05)
        print(f"Saved: {output_file}")

    plt.close()


# ============================================================
# RUN
# ============================================================

display_df = prepare_table(input_df)
display_rows, group_start_rows = build_display_rows(display_df)
save_table_image(display_rows, group_start_rows)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# LOAD DATA
# ============================================================

df_cohorts = pd.read_csv("/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/participants.csv") 
df_cohorts = df_cohorts.iloc[1:].reset_index(drop=True)

# ============================================================
# EDIT THIS SECTION
# ============================================================

input_df = df_cohorts
output_file = "ndd_cohort_sex_table.png"
preview_only = False   # True = show on screen, False = save to file

ndd_col = "NDD"

cohort_groups = [
    ("FinnGen", ["Female", "Male", "All"]),
    ("UKB", ["Female", "Male", "All"]),
    ("AoU", ["Female", "Male", "All"])
]

cohort_display_names = {
    "AoU": "All of Us",
    "FinnGen": "FinnGen",
    "UKB": "UK Biobank",
}
figure_width = 24
row_height = 0.90
font_size = 20
dpi = 600


# ============================================================
# HELPERS
# ============================================================

def clean_blank_cells(series):
    return series.replace(r"^\s*$", np.nan, regex=True)


def format_count(x):
    if pd.isna(x) or str(x).strip() == "":
        return ""
    try:
        return f"{int(float(x)):,}"
    except Exception:
        return str(x)


def prepare_table(df):
    df = df.copy()

    # Keep only the 10 visible columns from the sheet
    df = df.iloc[:, :10].copy()

    # Rename by position so there are no Unnamed columns
    df.columns = [
        "NDD",
        "AoU Female", "AoU Male", "AoU All",
        "FinnGen Female", "FinnGen Male", "FinnGen All",
        "UKB Female", "UKB Male", "UKB All",
    ]
    df.columns = df.columns.astype(str).str.strip()

    if ndd_col not in df.columns:
        raise KeyError(f"Missing required column: {ndd_col}")

    # Fill down repeated NDD labels
    df[ndd_col] = clean_blank_cells(df[ndd_col]).ffill()

    # Drop totally empty rows
    df = df.dropna(how="all").copy()

    value_cols = [
        f"{cohort} {sex}"
        for cohort, sexes in cohort_groups
        for sex in sexes
    ]

    missing = [c for c in value_cols if c not in df.columns]
    if missing:
        print("Missing columns:")
        for c in missing:
            print(f"  - {c}")

        print("\nAvailable columns:")
        for c in df.columns:
            print(f"  - {c}")

        raise ValueError("Your dataframe columns do not match the expected cohort/sex layout.")

    # Keep only needed columns
    display_df = df[[ndd_col] + value_cols].copy()

    # Format counts
    for c in value_cols:
        display_df[c] = display_df[c].apply(format_count)

    return display_df


def build_display_rows(display_df):
    display_rows = []
    previous_ndd = None

    for _, row in display_df.iterrows():
        ndd = str(row[ndd_col])
        show_ndd = ndd != previous_ndd

        row_out = [ndd if show_ndd else ""]

        for cohort, sexes in cohort_groups:
            for sex in sexes:
                row_out.append(row[f"{cohort} {sex}"])

        display_rows.append(row_out)
        previous_ndd = ndd

    return display_rows


def save_table_image(display_rows):
    # Two header rows
    header_top = [""]
    header_bottom = [ndd_col]

    for cohort, sexes in cohort_groups:
        header_top.extend(["", "", ""])
        header_bottom.extend(sexes)

    table_rows = [header_top, header_bottom] + display_rows

    n_rows = len(table_rows)
    n_cols = len(table_rows[0])

    fig_height = max(2.6, row_height * (len(display_rows) + 2.2))
    fig, ax = plt.subplots(figsize=(figure_width, fig_height))
    ax.axis("off")

    table = ax.table(
        cellText=table_rows,
        cellLoc="center",
        colLoc="center",
        loc="upper left",
        bbox=[0.01, 0.02, 0.98, 0.96],
    )

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.scale(1, 1.12)

    # Column widths
    col_widths = [0.14]
    cohort_widths = [0.095, 0.095, 0.085]
    for _ in cohort_groups:
        col_widths.extend(cohort_widths)

    for col_idx, width in enumerate(col_widths):
        for row_idx in range(n_rows):
            table[row_idx, col_idx].set_width(width)

    # Style cells
    for (row_idx, col_idx), cell in table.get_celld().items():
        cell.PAD = 0.03
        cell.set_edgecolor("white")
        cell.set_linewidth(0.0)

        if row_idx == 0:
            cell.set_facecolor("white")
            cell.set_text_props(weight="bold", color="#1F2937")
        elif row_idx == 1:
            cell.set_facecolor("white")
            cell.set_text_props(weight="bold", color="#1F2937")
        else:
            cell.set_facecolor("white")
            cell.set_text_props(color="#1F2937")

            # Bold NDD labels only once per block
            data_row_idx = row_idx - 2
            if col_idx == 0 and display_rows[data_row_idx][0] != "":
                cell.set_text_props(weight="bold", color="#1F2937")

    # Draw clean horizontal rules only
    fig.canvas.draw()

    left_x = table[0, 0].get_x()
    right_x = table[0, n_cols - 1].get_x() + table[0, n_cols - 1].get_width()

    top_y = table[0, 0].get_y() + table[0, 0].get_height()
    header_bottom_y = table[1, 0].get_y()
    bottom_y = table[n_rows - 1, 0].get_y()

    ax.hlines(top_y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.1)
    ax.hlines(bottom_y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.1)
    ax.hlines(header_bottom_y, left_x, right_x, transform=ax.transAxes, color="#667085", linewidth=1.0)

    # Cohort separators between NDD and each cohort block
    for sep_col in [1, 4, 7]:
        if sep_col < n_cols:
            x = table[0, sep_col].get_x()
            ax.vlines(x, bottom_y, top_y, transform=ax.transAxes, color="#98A2B3", linewidth=0.9)

    # Cohort labels centered across their 3-column spans
    start_col = 1
    for cohort, sexes in cohort_groups:
        end_col = start_col + 2
        x0 = table[0, start_col].get_x()
        x1 = table[0, end_col].get_x() + table[0, end_col].get_width()
        y0 = table[0, start_col].get_y()
        h = table[0, start_col].get_height()

        ax.text(
            (x0 + x1) / 2,
            y0 + h / 2,
            cohort_display_names.get(cohort, cohort),
            ha="center",
            va="center",
            fontsize=font_size,
            fontweight="bold",
            transform=ax.transAxes,
            color="#1F2937",
        )

        start_col += 3

    if preview_only:
        plt.show()
    else:
        output_file = "/home/jupyter/workspace/WORKSPACE_BUCKET/data/results/Tables/table_cohorts"
        plt.savefig(output_file, dpi=dpi, bbox_inches="tight", pad_inches=0.05)
        print(f"Saved: {output_file}")

    plt.close()


# ============================================================
# RUN
# ============================================================

display_df = prepare_table(input_df)
display_rows = build_display_rows(display_df)
save_table_image(display_rows)